# Suite2 Project10 Ann Cnn

**Dataset:** Fashion-MNIST (10,000 samples, 28×28)

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 10: Image Classification — ANN vs CNN"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

OUTPUT_DIR = os.path.join(BASE_DIR, 'suite2_project10_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 10: IMAGE CLASSIFICATION — ANN vs CNN

In [ ]:
# ── 1. Load Fashion-MNIST ──

## 1. Loading Fashion-MNIST Dataset

In [ ]:
# Download from OpenML
print("Downloading Fashion-MNIST... (may take a moment)")
fashion_mnist = fetch_openml('Fashion-MNIST', version=1, parser='auto')
X_full = fashion_mnist.data.values.astype(np.float32)
y_full = fashion_mnist.target.values.astype(int)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"\nDataset shape: {X_full.shape[0]:,} images × {X_full.shape[1]} pixels")
print(f"Classes: {len(class_names)}")
for i, name in enumerate(class_names):
    cnt = (y_full == i).sum()
    print(f"  {i}: {name:15s} — {cnt:,} images")

# Use subset for speed
np.random.seed(42)
idx = np.random.choice(len(X_full), 10000, replace=False)
X_subset = X_full[idx] / 255.0  # Normalize to [0, 1]
y_subset = y_full[idx]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_subset, y_subset, test_size=0.2, random_state=42)
print(f"\nTraining set: {X_train.shape[0]:,} images")
print(f"Test set:     {X_test.shape[0]:,} images")

# ── 2. Sample Images ──

## 2. Sample Images

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    ax = axes[i // 5, i % 5]
    idx_sample = np.where(y_train == i)[0][0]
    ax.imshow(X_train[idx_sample].reshape(28, 28), cmap='gray')
    ax.set_title(class_names[i], fontsize=10)
    ax.axis('off')
plt.suptitle('Fashion-MNIST — Sample Images (One per Class)', fontsize=14, y=1.02)
plt.tight_layout()
# 'p10_sample_images')
plt.show()
print("[Saved] p10_sample_images.png")

# ── 3. Baseline: Logistic Regression ──

## 3. Baseline: Logistic Regression (Pixel Input)

In [ ]:
lr = LogisticRegression(max_iter=1000, C=10.0, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f"Test accuracy: {acc_lr:.4f}")

# ── 4. ANN: Multi-layer Perceptron ──

## 4. ANN — Multi-Layer Perceptron

In [ ]:
ann = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu', 
                    max_iter=50, random_state=42, verbose=False)
ann.fit(X_train, y_train)
y_pred_ann = ann.predict(X_test)
acc_ann = accuracy_score(y_test, y_pred_ann)
print(f"ANN (128→64 neurons) Test Accuracy: {acc_ann:.4f}")

# Training curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ann.loss_curve_, linewidth=2, color='#2e86de')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('ANN Training Loss Curve', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p10_ann_loss')
plt.show()
print("[Saved] p10_ann_loss.png")

# ── 5. ANN vs CNN Conceptual ──

## 5. ANN vs CNN — Why Convolution Works

In [ ]:
print("""
=== ANN Architecture (Flatten → Dense) ===
Input: 784 pixels (28×28 flattened)
Layer 1: Dense(128, ReLU)     — 100,352 parameters
Layer 2: Dense(64, ReLU)      — 8,256 parameters
Output:  Dense(10, Softmax)    — 650 parameters
Total:   ~109,258 parameters

Problem: All spatial information is lost when flattening.
A shirt rotated 10° has completely different pixel values.

=== CNN Architecture (Convolutional) ===
Input: 28×28×1 (keeps 2D structure)
Conv 1: 32 filters, 3×3 kernel   — 320 parameters
Pool 1: 2×2 max pooling
Conv 2: 64 filters, 3×3 kernel   — 18,496 parameters  
Pool 2: 2×2 max pooling
Flatten → Dense(128) → Output(10)
Total:   ~25,306 parameters (4× fewer than ANN!)

Key insight: CNNs preserve spatial relationships and share weights
across the image → fewer parameters, better generalization.
""")

# ── 6. Confusion Matrix for ANN ──

## 6. ANN Classification Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ann, ax=ax, cmap='Blues', 
                                         display_labels=class_names, xticks_rotation=45)
ax.set_title(f'ANN Confusion Matrix (Accuracy = {acc_ann:.3f})', fontsize=14)
plt.tight_layout()
# 'p10_ann_confusion')
plt.show()
print("[Saved] p10_ann_confusion.png")

print(f"\n=== ANN Classification Report ===")
print(classification_report(y_test, y_pred_ann, target_names=class_names, digits=3))

# ── 7. Visualize Misclassifications ──

## 7. Misclassified Examples

In [ ]:
misclassified = np.where(y_pred_ann != y_test)[0][:10]
fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for i, idx_m in enumerate(misclassified):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_test[idx_m].reshape(28, 28), cmap='gray')
    ax.set_title(f'True: {class_names[y_test[idx_m]]}\\nPred: {class_names[y_pred_ann[idx_m]]}', 
                 fontsize=8, color='red')
    ax.axis('off')
plt.suptitle('ANN Misclassifications', fontsize=14, y=1.02)
plt.tight_layout()
# 'p10_misclassified')
plt.show()
print("[Saved] p10_misclassified.png")

results = {
    'dataset': 'Fashion-MNIST',
    'n_samples': 10000,
    'n_classes': 10,
    'class_names': class_names,
    'logistic_regression_accuracy': float(acc_lr),
    'ann_architecture': [128, 64],
    'ann_accuracy': float(acc_ann),
    'ann_training_iterations': len(ann.loss_curve_),
    'ann_final_loss': float(ann.loss_curve_[-1]),
}
    json.dump(results, f, indent=2, default=str)

print("Done.")